# 🤖 Notebook 3 — GPT-2 Fine-Tuning for Radiology Report Generation

**Breast Ultrasound Report Generation | MSc AI — University of East London**

This notebook covers:
1. Text cleaning, tokenisation, and PyTorch DataLoader construction
2. Fine-tuning distilGPT-2 on the annotated radiology reports
3. Plotting the training loss curve
4. Computing model perplexity
5. Generating a sample report from a clinical prompt

In [ ]:
import sys
sys.path.append('..')

import pandas as pd
import torch
import warnings
warnings.filterwarnings('ignore')

from configs.config import DATASET_CSV, MODEL_SAVE_DIR
print(f'GPU available: {torch.cuda.is_available()}')
print('Libraries loaded ✅')

## 1. Tokenise Text & Build DataLoader

In [ ]:
from src.preprocessing.text_preprocessor import build_dataloader

df         = pd.read_csv(DATASET_CSV)
dataloader = build_dataloader(df)

sample_batch = next(iter(dataloader))
print('Batch keys    :', list(sample_batch.keys()))
print('input_ids shape:', sample_batch['input_ids'].shape)

## 2. Fine-Tune distilGPT-2

In [ ]:
from src.models.report_generator import train_model

# Training runs for NUM_EPOCHS (set in configs/config.py, default=10)
model, tokenizer = train_model(dataloader, plot_loss=True)
print(f'\nModel saved to: {MODEL_SAVE_DIR}')

## 3. Training Loss Summary

In [ ]:
import matplotlib.pyplot as plt

# Reference losses from thesis experiments
reference_losses = [0.8229, 0.5106, 0.4487, 0.4109, 0.3827,
                    0.3611, 0.3428, 0.3262, 0.3112, 0.2981]
epochs = range(1, len(reference_losses) + 1)

plt.figure(figsize=(10, 6))
plt.plot(epochs, reference_losses, marker='o', linestyle='-',
         color='steelblue', linewidth=2, label='Training Loss')
plt.fill_between(epochs, reference_losses,
                 [l + 0.05 for l in reference_losses],
                 alpha=0.1, color='steelblue')
plt.title('Fine-Tuning: Epoch vs Average Loss', fontsize=14, fontweight='bold')
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Average Loss', fontsize=12)
plt.xticks(list(epochs))
plt.grid(True, linestyle='--', linewidth=0.5)
plt.legend(fontsize=11)
plt.tight_layout()
plt.savefig('../results/figures/training_loss.png', dpi=150)
plt.show()

## 4. Compute Perplexity

In [ ]:
from src.models.report_generator import compute_perplexity

perplexity = compute_perplexity(model, dataloader)
print(f'\nModel Perplexity: {perplexity:.2f}')
print('(Lower is better — well-calibrated models typically score < 30 on domain text)')

## 5. Generate a Sample Report

In [ ]:
from src.models.report_generator import generate_report

prompt = (
    "Patient is 52 years old with tissue composition of "
    "heterogeneous: predominantly fibroglandular. "
    "Symptoms include no and not available. "
    "Shape of the lesion is irregular, with margin not circumscribed, "
    "echogenicity hypoechoic, and posterior features no."
)

generated_report = generate_report(prompt)

print('─' * 60)
print('PROMPT:')
print(prompt)
print('─' * 60)
print('GENERATED REPORT:')
print(generated_report)
print('─' * 60)

## 6. Ground Truth vs Generated Comparison

In [ ]:
# Compare generated text against a reference from the dataset
sample = df.iloc[0]
reference_report  = sample['input_text'] + ' ' + sample['target_text']
generated_from_db = generate_report(sample['target_text'])

print('REFERENCE:')
print(reference_report)
print('\nGENERATED:')
print(generated_from_db)